# `LayerNorm` 前端支持

## {func}`~tvm.relax.frontend.torch.from_exported_program`

In [1]:
import torch
from torch.export import export
import tvm
from tvm import relax
import tvm.testing
from tvm.script import ir as I
from tvm.script import relax as R
from tvm.script import tir as T
from tvm.relax.frontend.torch import from_exported_program
class LayerNorm(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.ln = torch.nn.LayerNorm((10, 10))

    def forward(self, x):
        return self.ln(x)

In [2]:
example_args = (torch.randn(1, 3, 10, 10, dtype=torch.float32),)

model = LayerNorm()
exported_program = export(model, args=example_args,)
mod = from_exported_program(exported_program)
mod.show()

## {class}`~tvm.relax.frontend.nn.modules.LayerNorm`

In [3]:
from tvm.relax.frontend.nn import core, modules, spec
mod = modules.LayerNorm(8)
tvm_mod, params = mod.export_tvm(
    spec={"forward": {"x": spec.Tensor((2, 4, 8), "float32")}}, debug=True
)
tvm_mod.show()

## {func}`~tvm.relax.frontend.torch.from_fx`

In [4]:
import tvm
from tvm import relax
import tvm.testing
from tvm.script import ir as I
from tvm.script import relax as R
from tvm.script import tir as T
from tvm.relax.frontend import detach_params
from tvm.relax.frontend.torch import from_fx

In [5]:
class LayerNorm(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.ln = torch.nn.LayerNorm((10, 10))

    def forward(self, x):
        return self.ln(x)

input_info = [([1, 3, 10, 10], "float32")]

torch_model = LayerNorm()
graph_model = torch.fx.symbolic_trace(torch_model)
with torch.no_grad():
    mod = from_fx(graph_model, input_info)
    mod.show()

In [6]:
class LayerNorm(torch.nn.Module):
    def __init__(self, shape):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.ones(shape))
        self.bias = torch.nn.Parameter(torch.zeros(shape))

    def forward(self, x):
        return torch.nn.functional.layer_norm(
            x, self.weight.shape, self.weight, self.bias, 1e-5
        )

torch_model = LayerNorm((10, 10))
graph_model = torch.fx.symbolic_trace(torch_model)
with torch.no_grad():
    mod = from_fx(graph_model, input_info)
    mod.show()

In [7]:
class LayerNorm2(torch.nn.Module):
    def __init__(self, shape):
        super().__init__()
        self.shape = shape
        self.weight = None
        self.bias = None

    def forward(self, input):
        return torch.nn.functional.layer_norm(input, self.shape, self.weight, self.bias, 1e-5)

torch_model = LayerNorm2((10, 10))
graph_model = torch.fx.symbolic_trace(torch_model)
with torch.no_grad():
    mod = from_fx(graph_model, input_info)
    mod.show()

In [8]:
class LayerNorm3(torch.nn.Module):
    def __init__(self, shape):
        super().__init__()
        self.shape = shape
        self.weight = torch.nn.Parameter(torch.ones(shape))
        self.bias = torch.nn.Parameter(torch.zeros(shape))

    def forward(self, x):
        return torch.nn.functional.layer_norm(x, self.shape, self.weight, self.bias, 1e-5)

torch_model = LayerNorm3((10, 10))
graph_model = torch.fx.symbolic_trace(torch_model)
with torch.no_grad():
    mod = from_fx(graph_model, input_info)
    mod.show()